# NaniGPT — Day 11: Gradio demo

**Goal:** spin up a public-URL demo of NaniGPT that a real caregiver can try in 5 minutes. Three tabs:
1. **Daily check-in** — upload pill photo OR type/speak a journal entry → agent dispatches tools live
2. **Today's log** — chronological view of all log entries
3. **Doctor visit prep** — generate the report for the next appointment

**Pre-req:** `model`, `tokenizer`, `TOOLS`, `TOOL_MAP`, `_LOG`, and `parse_gemma_tool_calls` must already be defined in this Colab session (from notebooks 01 and 02).


## Step 1 — Install Gradio


In [ ]:
!pip install -qqq gradio>=5.0

## Step 2 — Helpers: classify a pill photo, run the agent, render the log


In [ ]:
import json, re, torch
from PIL import Image

# Persistent state for the diff: yesterday's classification keyed by 'last_seen'
_PILL_STATE = {'yesterday': None}

CLASSIFY_PROMPT = '''List EVERY compartment in this weekly pill organizer (MON-SUN, AM and PM = 14 total). For each, output JSON: [{"day":"MON","ampm":"AM","has_pills":true}, ...]. Output ONLY the JSON array, nothing else.'''

def classify_organizer(img):
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': 'You are NaniGPT, a private companion for caregivers.'}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': CLASSIFY_PROMPT},
        ]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=600, temperature=0.2, top_p=0.9, do_sample=False)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'\[.*\]', text, re.DOTALL)
    return json.loads(m.group(0)) if m else None

def diff_states(yesterday, today):
    if not yesterday or not today: return []
    yest = {(s['day'], s['ampm']): s['has_pills'] for s in yesterday}
    todo = {(s['day'], s['ampm']): s['has_pills'] for s in today}
    changes = []
    for slot, was in yest.items():
        now = todo.get(slot, was)
        if was and not now:
            changes.append({'day': slot[0], 'ampm': slot[1], 'change': 'taken'})
        elif not was and now:
            changes.append({'day': slot[0], 'ampm': slot[1], 'change': 'refilled'})
    return changes

## Step 3 — The Gradio app


In [ ]:
import gradio as gr
from datetime import datetime

AGENT_SYSTEM = '''You are NaniGPT, a private companion for adult children caring for aging parents with dementia. When the caregiver tells you something, decide whether to call one or more tools to log it, queue it for the doctor, or notify other family. Use tools liberally — that's the point. Reply briefly after the tool calls to confirm what you did.'''

def run_agent_for_ui(user_message):
    if not user_message or not user_message.strip():
        return 'Tell me what happened today.', []
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': AGENT_SYSTEM}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': user_message}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tools=TOOLS, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=600, temperature=0.3, top_p=0.9, do_sample=False)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    calls = parse_gemma_tool_calls(raw, TOOL_MAP)
    summaries = []
    for name, args in calls:
        try:
            r = TOOL_MAP[name](**args)
            summaries.append(f'  ✓ {name}({args})')
        except Exception as e:
            summaries.append(f'  ✗ {name}: {e}')
    # Strip out the tool calls from the raw to leave just the model's reply
    reply = re.sub(r'call:\w+\{.*?\}', '', raw, flags=re.DOTALL).strip() or 'Logged.'
    return reply, '\n'.join(summaries) if summaries else '(no tool calls)'

def handle_pill_photo(img):
    if img is None:
        return 'Upload a photo to begin.', '', ''
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    img = img.convert('RGB')
    img.thumbnail((768, 768))

    today = classify_organizer(img)
    today_str = json.dumps(today, indent=2) if today else '(failed to parse)'

    yesterday = _PILL_STATE['yesterday']
    if yesterday is None:
        _PILL_STATE['yesterday'] = today
        return today_str, '(no prior photo to compare to — saved as baseline)', '(no changes to log)'

    changes = diff_states(yesterday, today)
    _PILL_STATE['yesterday'] = today  # Update for next call

    if not changes:
        return today_str, 'No compartments changed since the last photo.', '(no changes to log)'

    # Auto-log each change
    for c in changes:
        TOOL_MAP['log_pill_change'](day=c['day'], ampm=c['ampm'], change=c['change'])

    change_lines = '\n'.join(f"  • {c['day']} {c['ampm']}: {c['change']}" for c in changes)
    return today_str, f'{len(changes)} change(s) detected:\n{change_lines}', f'Logged {len(changes)} entries to caregiver log.'

def render_log(category_filter):
    entries = _LOG if category_filter == 'all' else [e for e in _LOG if e.get('category') == category_filter]
    if not entries:
        return '(no entries yet)'
    lines = []
    for e in entries:
        ts = e['ts'][11:16]  # HH:MM
        cat = e.get('category', '?')
        if cat == 'medication':
            lines.append(f"[{ts}] 💊 {e.get('day')} {e.get('ampm')} — {e.get('change')}")
        elif cat == 'incident':
            sev = {'low':'🟢','medium':'🟡','high':'🔴'}.get(e.get('severity'), '⚪')
            lines.append(f"[{ts}] {sev} {e.get('incident_type','?')}: {e.get('detail','')[:80]}")
        elif cat == 'doctor_agenda':
            lines.append(f"[{ts}] 📋 doctor agenda ({e.get('priority')}): {e.get('item','')}")
        elif cat == 'sibling_notify':
            lines.append(f"[{ts}] 📨 sibling ({e.get('urgency')}): {e.get('message','')}")
        else:
            lines.append(f"[{ts}] {e}")
    return '\n'.join(lines)

def make_doctor_report(patient_name, days):
    if not patient_name.strip():
        return '(enter a patient name)', ''
    result = TOOL_MAP['generate_doctor_pdf'](patient_name=patient_name, days=int(days))
    summary = '\n'.join([
        f"Patient: {result['patient']}",
        f"Period: last {result['days']} days",
        f"Total log entries: {result['entries_count']}",
        f"PDF path (stub): {result['path']}",
    ])
    # Render the entries that would go in the PDF
    return summary, render_log('all')

with gr.Blocks(title='NaniGPT', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# NaniGPT\n*A private companion for adult children caring for aging parents with dementia. All AI runs on-device. No cloud. No data leaves your phone.*')

    with gr.Tabs():
        # ---- Tab 1: Daily check-in ----
        with gr.Tab('Daily check-in'):
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### 📷 Pill organizer photo')
                    pill_img = gr.Image(type='pil', label='Photograph the weekly organizer')
                    pill_btn = gr.Button('Analyze pill organizer', variant='primary')
                    pill_today_json = gr.Textbox(label="Today's classification (JSON)", lines=8)
                    pill_changes = gr.Textbox(label='Changes since last photo', lines=4)
                    pill_logged = gr.Textbox(label='Logged to caregiver log', lines=2)
                    pill_btn.click(handle_pill_photo, inputs=pill_img, outputs=[pill_today_json, pill_changes, pill_logged])

                with gr.Column():
                    gr.Markdown('### 💬 Talk to NaniGPT\nDescribe anything that happened. NaniGPT will log it, queue items for the doctor, and notify family if relevant.')
                    journal_input = gr.Textbox(label='What happened today?', placeholder="e.g. 'Mom slept 3 hours, refused breakfast, asked where Dad is three times.'", lines=4)
                    journal_btn = gr.Button('Log it', variant='primary')
                    journal_reply = gr.Textbox(label='NaniGPT', lines=3)
                    journal_calls = gr.Textbox(label='Tool calls executed', lines=4)
                    journal_btn.click(run_agent_for_ui, inputs=journal_input, outputs=[journal_reply, journal_calls])

        # ---- Tab 2: Today's log ----
        with gr.Tab("Today's log"):
            cat_filter = gr.Radio(
                ['all', 'medication', 'incident', 'doctor_agenda', 'sibling_notify'],
                value='all', label='Filter'
            )
            log_view = gr.Textbox(label='Caregiver log', lines=20)
            refresh_btn = gr.Button('Refresh')
            cat_filter.change(render_log, inputs=cat_filter, outputs=log_view)
            refresh_btn.click(render_log, inputs=cat_filter, outputs=log_view)

        # ---- Tab 3: Doctor visit prep ----
        with gr.Tab('Doctor visit prep'):
            gr.Markdown('### Print the report for the next appointment')
            with gr.Row():
                patient_in = gr.Textbox(label='Patient name', value='Mom')
                days_in = gr.Number(label='Days to include', value=30, precision=0)
            report_btn = gr.Button('Generate report', variant='primary')
            report_summary = gr.Textbox(label='Report summary', lines=5)
            report_body = gr.Textbox(label='Report contents', lines=20)
            report_btn.click(make_doctor_report, inputs=[patient_in, days_in], outputs=[report_summary, report_body])

demo.launch(share=True)

## What you should see

After the cell runs, Gradio prints a **public share URL** like `https://abc123-xyz.gradio.live`. That URL lets anyone (your real caregiver interviewee, judges previewing your work, you on your phone) try the app for ~72 hours.

**Demo flow that proves NaniGPT works end-to-end:**
1. Tab 1 → upload `test2_baseline_full_week.jpg` → 'no prior photo, saved as baseline'
2. Tab 1 → upload `test3_three_taken_today.jpg` → '3 changes detected: MON AM taken, MON PM taken, TUE AM taken'
3. Tab 1 → in the journal box, type: 'Mom asked where dad is three times tonight, refused dinner, has a small bruise on her forearm.'
4. Tab 2 → see all 6+ entries logged with proper icons + timestamps
5. Tab 3 → click Generate → see the 30-day summary that would go to the doctor

If all 5 steps work, **the demo is shippable for the video.**
